# Phase 3 — QLoRA Fine-Tuning — intent-classifier

This notebook runs **QLoRA** (4-bit NF4 quantized LoRA) on the same models and hyperparameters
as `finetune_LoRA`, enabling direct comparison of accuracy and memory usage.

1. Clone the `intent-classifier` repo from GitHub
2. Install required Python packages (peft, trl, bitsandbytes, etc.)
3. Authenticate with Hugging Face
4. Check GPU environment
5. Prepare train / val / test JSONL splits
6. Run a 10-step smoke test to validate the pipeline
7. Train QLoRA adapters for selected models and configs
8. Evaluate on the validation set
9. Evaluate on the test set (locked — run deliberately)
10. Run inference sanity checks on hand-picked examples
11. Generate analysis plots
12. Display results inline
13. Download outputs as a zip archive

> **Runtime:** Set runtime type to **GPU → L4** (or A100) for training.  
> 4-bit QLoRA uses ~50% less VRAM than full LoRA, so a T4 can handle configs A and B on 1k data.
>
> **bitsandbytes** requires CUDA — MPS is not supported for 4-bit training.


## 1. Clone repository

In [ ]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes.")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")


## 2. Install dependencies

In [ ]:
%pip install -q \
    torch \
    transformers \
    accelerate \
    peft \
    trl \
    datasets \
    bitsandbytes \
    python-dotenv \
    sentencepiece \
    protobuf \
    matplotlib \
    numpy \
    "torchao>=0.16.0"

import pkg_resources
for pkg in ["torch", "transformers", "peft", "trl", "datasets", "bitsandbytes", "torchao"]:
    try:
        v = pkg_resources.get_distribution(pkg).version
        print(f"  {pkg:<18} {v}")
    except Exception:
        print(f"  {pkg:<18} NOT FOUND")


## 3. Hugging Face authentication

Both Qwen2.5-0.5B-Instruct and Qwen3-0.6B are publicly available and do not require a token.
Set `HF_TOKEN` as a Colab secret if you later add gated models.


In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("HF_TOKEN secret not set — public models only.")
except Exception:
    print("Not running in Colab or no secrets available.")


## 4. GPU environment check

In [ ]:
import torch, platform

print(f"Python     : {platform.python_version()}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    dev = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(dev)
    print(f"GPU        : {props.name}  (device {dev})")
    print(f"VRAM       : {props.total_memory / 1024**3:.1f} GB")
    print(f"BF16       : {torch.cuda.is_bf16_supported()}")
    cap = f"{props.major}.{props.minor}"
    print(f"Compute cap: {cap}")
else:
    print("No CUDA GPU detected — 4-bit quantization will not be available.")


## 5. Configuration

Edit the variables below to control which models and configs are trained.


In [ ]:
import sys

REPO_DIR  = "/content/intent-classifier"
SRC_DIR   = f"{REPO_DIR}/finetune_QLoRA/src"
DATA_DIR  = f"{REPO_DIR}/dataset_full"
SPLIT_DIR = f"{REPO_DIR}/finetune_QLoRA/data"
CKPT_DIR  = f"{REPO_DIR}/finetune_QLoRA/checkpoints"
PLOT_DIR  = f"{REPO_DIR}/finetune_QLoRA/analysis"

sys.path.insert(0, SRC_DIR)

# ── Dataset size toggle ───────────────────────────────────────────────────
# "1k"  — first 10 files (1 000 examples): fast calibration, ~3-5 min/run on L4
# "10k" — all 100 files (10 000 examples): full scale,       ~30-50 min/run on L4
DATASET_SIZE = "1k"

# ── Models and configs to run ─────────────────────────────────────────────
MODELS_TO_RUN  = ["qwen2.5-0.5b", "qwen3-0.6b"]
CONFIGS_TO_RUN = ["A", "B", "C"]   # A=light, B=standard, C=wide

DEVICE = "auto"   # auto | cuda | cpu

print(f"Dataset size  : {DATASET_SIZE}")
print(f"Models        : {MODELS_TO_RUN}")
print(f"QLoRA configs : {CONFIGS_TO_RUN}")
print(f"Device        : {DEVICE}")
print(f"Total runs    : {len(MODELS_TO_RUN) * len(CONFIGS_TO_RUN)}")


## 6. (Optional) Mount Google Drive for persistent checkpoints

Skip this cell if you do not need to persist checkpoints across sessions.


In [ ]:
# Uncomment the lines below to mount Drive and redirect checkpoints there.

# from google.colab import drive
# drive.mount('/content/drive')
# CKPT_DIR = "/content/drive/MyDrive/intent-classifier/finetune_QLoRA/checkpoints"
# print(f"Checkpoints will be saved to: {CKPT_DIR}")


## 7. Prepare data splits

In [ ]:
import subprocess, sys, os

os.makedirs(SPLIT_DIR, exist_ok=True)

result = subprocess.run(
    [
        sys.executable,
        f"{SRC_DIR}/prepare_qlora_data.py",
        "--dataset-size", DATASET_SIZE,
        "--data-dir",     DATA_DIR,
        "--out-dir",      SPLIT_DIR,
    ],
    check=True, text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)


## 8. Smoke test

Runs **10 training steps** on one model + config to validate the full pipeline  
(data loading → tokenisation → 4-bit quantization → LoRA → SFTTrainer → callback → report write)  
without committing to a full training run.

Expected runtime: **< 2 minutes** on L4.


In [ ]:
import subprocess, sys, time

SMOKE_MODEL  = MODELS_TO_RUN[0]   # test with the first model in the list
SMOKE_CONFIG = "A"                # lightest config for fastest smoke test

print(f"Smoke test — model={SMOKE_MODEL}  config={SMOKE_CONFIG}  dataset={DATASET_SIZE}\n")
t0 = time.time()

result = subprocess.run(
    [
        sys.executable,
        f"{SRC_DIR}/qlora_train.py",
        "--model",        SMOKE_MODEL,
        "--lora-config",  SMOKE_CONFIG,
        "--dataset-size", DATASET_SIZE,
        "--ckpt-dir",     CKPT_DIR,
        "--device",       DEVICE,
        "--smoke-test",
    ],
    capture_output=True,
    text=True,
)

elapsed = time.time() - t0

if result.stdout.strip():
    print(result.stdout)

if result.returncode == 0:
    print(f"\n\u2705 Smoke test PASSED  ({elapsed:.0f}s)")
else:
    print(f"\n\u274c Smoke test FAILED  (exit code {result.returncode})")
    if result.stderr.strip():
        print("\n\u2500\u2500 stderr " + "\u2500" * 50)
        print(result.stderr)
        print("\u2500" * 59)
    raise RuntimeError(f"Smoke test failed (exit code {result.returncode})")


## 9. Train

Runs all (model × config) combinations sequentially.  
Estimated runtime per run (L4, 1k dataset): Config A ≈ 3 min, B ≈ 5 min, C ≈ 8 min.


In [ ]:
import subprocess, sys, time

TRAIN_SCRIPT = f"{SRC_DIR}/run_qlora_experiments.py"

cmd = [
    sys.executable, TRAIN_SCRIPT,
    "--models",       *MODELS_TO_RUN,
    "--configs",      *CONFIGS_TO_RUN,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
]

print("Starting training...\n")
t0 = time.time()
result = subprocess.run(cmd, text=True)
elapsed = time.time() - t0

if result.returncode == 0:
    print(f"\n\u2705 Training complete  ({elapsed:.0f}s  /  {elapsed/60:.1f} min)")
else:
    print(f"\n\u274c Training failed  (exit code {result.returncode})")


## 10. Validate (validation set)

Loads each trained adapter and evaluates on the validation set.


In [ ]:
import subprocess, sys

EVAL_SCRIPT = f"{SRC_DIR}/run_qlora_experiments.py"

cmd = [
    sys.executable, EVAL_SCRIPT,
    "--models",        *MODELS_TO_RUN,
    "--configs",       *CONFIGS_TO_RUN,
    "--dataset-size",  DATASET_SIZE,
    "--device",        DEVICE,
    "--skip-training",
]

result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    print(f"\n\u274c Validation failed  (exit code {result.returncode})")


## 11. Test evaluation (locked)

Run this cell deliberately once you are satisfied with the validation results.  
Evaluating the test set repeatedly can introduce selection bias.


In [ ]:
import subprocess, sys

VALIDATE_SCRIPT = f"{SRC_DIR}/qlora_validate.py"

for model_key in MODELS_TO_RUN:
    for cfg in CONFIGS_TO_RUN:
        print(f"\n\u2500\u2500 Test eval: {model_key} / config-{cfg} \u2500\u2500")
        result = subprocess.run(
            [
                sys.executable, VALIDATE_SCRIPT,
                "--model",        model_key,
                "--lora-config",  cfg,
                "--dataset-size", DATASET_SIZE,
                "--split",        "test",
                "--device",       DEVICE,
            ],
            text=True,
        )
        if result.returncode != 0:
            print(f"  \u274c FAILED  (exit code {result.returncode})")


## 12. Inference sanity check

Manually inspect model outputs on the first 5 examples from the test anchor.


In [ ]:
import json, sys, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

sys.path.insert(0, SRC_DIR)
from common import MODEL_REGISTRY, build_chat_messages, apply_chat_template_safe, extract_prediction  # type: ignore

SANITY_MODEL  = MODELS_TO_RUN[0]
SANITY_CONFIG = "B"

ckpt_path = Path(CKPT_DIR) / f"{SANITY_MODEL}_config_{SANITY_CONFIG}_{DATASET_SIZE}"
model_id  = MODEL_REGISTRY[SANITY_MODEL]
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading {SANITY_MODEL} + QLoRA config-{SANITY_CONFIG} from {ckpt_path}...")

bnb_config = None
if device.type == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

tokenizer = AutoTokenizer.from_pretrained(str(ckpt_path))
base      = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if device.type == "cuda" else torch.float32,
    device_map={"":device},
)
model     = PeftModel.from_pretrained(base, str(ckpt_path))
model.eval()
print("Model loaded.\n")

anchor_file = Path(SPLIT_DIR) / DATASET_SIZE / "test_anchor.jsonl"
sanity_examples = [
    json.loads(l) for l in anchor_file.read_text().splitlines() if l.strip()
][:5]

print(f"{'\u2500'*70}")
print(f"  {'REQUEST':<35}  {'EXPECTED':<20}  {'PREDICTED':<20}  OK")
print(f"{'\u2500'*70}")

for ex in sanity_examples:
    messages = build_chat_messages(ex, include_answer=False)
    text     = apply_chat_template_safe(tokenizer, messages, SANITY_MODEL, add_generation_prompt=True)
    inputs   = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=16, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    pred    = extract_prediction(tokenizer.decode(new_ids, skip_special_tokens=True))
    ok      = "\u2705" if pred == ex["answer"] else "\u274c"
    req     = ex["user_request"][:33]
    print(f"  {req:<35}  {ex['answer']:<20}  {pred:<20}  {ok}")


## 13. Generate analysis plots

In [ ]:
import subprocess, sys, os

os.makedirs(PLOT_DIR, exist_ok=True)

PLOT_SCRIPT = f"{SRC_DIR}/plot_qlora_results.py"

result = subprocess.run(
    [
        sys.executable, PLOT_SCRIPT,
        "--out-dir", PLOT_DIR,
    ],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0 and result.stderr:
    print(result.stderr)


## 14. Display results

In [ ]:
from IPython.display import display, Image  # type: ignore
from pathlib import Path

for fname in ["qlora_accuracy_comparison.png", "qlora_training_curves.png", "qlora_memory_comparison.png"]:
    p = Path(PLOT_DIR) / fname
    if p.exists():
        print(f"\n{fname}")
        display(Image(str(p)))
    else:
        print(f"  (not found: {p})")


## 15. Download outputs (optional)

Zips and downloads:
- Training reports (`reports_training/`)
- Validation reports (`reports_validation/`)
- Analysis plots (`analysis/`)


In [ ]:
import shutil, os
from pathlib import Path
from google.colab import files  # type: ignore

REPO_DIR = "/content/intent-classifier"
zip_base = "/content/qlora_outputs"

shutil.make_archive(
    zip_base,
    "zip",
    root_dir=f"{REPO_DIR}/finetune_QLoRA",
    base_dir=".",
)

zip_path = zip_base + ".zip"
size_mb  = os.path.getsize(zip_path) / 1024**2
print(f"Archive: {zip_path}  ({size_mb:.1f} MB)")
files.download(zip_path)
